# Visualização Computacional

## O que é?

**Visualização Computacional** (ou Científica) é a área que usa recursos gráficos para **comunicar dados e fenômenos complexos de forma visual e intuitiva**. O fluxo é: **Dados → Mapeamento Visual → Representação Gráfica**.

## Características Principais

- **Entrada:** dados numéricos (escalares, vetores, tensores, séries temporais, grafos)
- **Saída:** gráficos, mapas, superfícies, animações interativas
- **Técnicas:** mapeamento de cores (colormaps), isosuperfícies, glifos vetoriais, volumes
- **Aplicações:** meteorologia, engenharia (CFD, FEM), medicina (PET, fMRI), astronomia, análise de dados

## Diferença Chave

Enquanto as outras áreas lidam com imagens como **objeto primário**, a Visualização usa imagens como **meio de comunicação de dados**:

| Área | Objeto primário | Objetivo |
|------|----------------|----------|
| Síntese de Imagens | Cena 3D | Gerar imagem realista |
| Processamento de Imagens | Imagem existente | Melhorar/transformar |
| Visão Computacional | Imagem real | Interpretar/classificar |
| **Visualização Computacional** | **Dados abstratos** | **Tornar visível o invisível** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import gridspec
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import warnings
warnings.filterwarnings('ignore')

## 1. Mapeamento de Cor (Colormap)

O **colormap** é a ferramenta fundamental da Visualização Computacional: mapeia valores escalares em cores, tornando campos invisíveis (temperatura, pressão, intensidade) perceptíveis ao olho humano.

A escolha do colormap afeta diretamente a percepção dos dados — um colormap inadequado pode criar estruturas visuais que não existem nos dados (*perceptual artifacts*).

In [ ]:
# Campo escalar: função de temperatura simulada
x = np.linspace(-3, 3, 300)
y = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x, y)
Z_temperatura = np.sin(X) * np.cos(Y) * np.exp(-0.2*(X**2 + Y**2)) + 0.3*np.sin(2*X+Y)

colormaps = [
    ('viridis',     'Viridis (perceptual uniforme — RECOMENDADO)'),
    ('plasma',      'Plasma (perceptual uniforme)'),
    ('RdBu_r',      'RdBu (divergente — ideal para dados com centro zero)'),
    ('jet',         'Jet (sequencial clássico — EVITAR: artefatos visuais)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (cmap, label) in zip(axes.flat, colormaps):
    im = ax.contourf(X, Y, Z_temperatura, levels=50, cmap=cmap)
    ax.contour(X, Y, Z_temperatura, levels=10, colors='white', alpha=0.3, linewidths=0.5)
    plt.colorbar(im, ax=ax, label='Temperatura (u.a.)')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.suptitle('Impacto do Colormap na Percepção de Dados\n(mesmos dados, mapas diferentes)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Superfícies 3D e Isocurvas

A superfície 3D e suas **isocurvas** (linhas de mesmo valor, como curvas de nível) são as representações mais usadas em física computacional, meteorologia e engenharia.

In [ ]:
fig = plt.figure(figsize=(16, 5))

# Superfície 3D
ax1 = fig.add_subplot(131, projection='3d')
surf = ax1.plot_surface(X[::5,::5], Y[::5,::5], Z_temperatura[::5,::5],
                         cmap='viridis', alpha=0.9, linewidth=0)
fig.colorbar(surf, ax=ax1, shrink=0.5, label='T')
ax1.set_title('Superfície 3D', fontsize=11)
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('T')

# Heatmap + isocurvas (mapa de contorno)
ax2 = fig.add_subplot(132)
im2 = ax2.imshow(Z_temperatura, extent=[-3,3,-3,3], origin='lower',
                  cmap='viridis', aspect='auto')
contours = ax2.contour(X, Y, Z_temperatura, levels=12, colors='white', linewidths=0.8)
ax2.clabel(contours, fontsize=7, fmt='%.1f')
plt.colorbar(im2, ax=ax2, label='T')
ax2.set_title('Heatmap + Isocurvas', fontsize=11)
ax2.set_xlabel('x'); ax2.set_ylabel('y')

# Wireframe
ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_wireframe(X[::8,::8], Y[::8,::8], Z_temperatura[::8,::8],
                    color='steelblue', linewidth=0.5, alpha=0.7)
ax3.set_title('Wireframe', fontsize=11)
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('T')

plt.suptitle('Representações de Campo Escalar 2D', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Visualização de Campo Vetorial

Campos vetoriais (velocidade de fluido, campo elétrico, força) são visualizados com **quiver plots** (glifos de seta) ou **streamlines** (linhas de fluxo).

Esta visualização simula um campo de velocidade de fluido em torno de um obstáculo circular.

In [ ]:
# Campo vetorial: fluxo de fluido potencial ao redor de um cilindro
xv = np.linspace(-3, 3, 40)
yv = np.linspace(-2, 2, 30)
Xv, Yv = np.meshgrid(xv, yv)

R = 0.5  # raio do cilindro
r2 = Xv**2 + Yv**2
r2 = np.where(r2 < R**2, R**2, r2)  # máscara dentro do cilindro

# Velocidade de fluxo potencial: U = (1 - R²/r²) î + ... (simplificado)
U = 1 - R**2 * (Xv**2 - Yv**2) / r2**2
V = -2 * R**2 * Xv * Yv / r2**2

# Máscara dentro do cilindro
mask_cilindro = (Xv**2 + Yv**2) < R**2
U[mask_cilindro] = 0
V[mask_cilindro] = 0

velocidade = np.sqrt(U**2 + V**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quiver plot
ax1 = axes[0]
im = ax1.contourf(Xv, Yv, velocidade, levels=30, cmap='RdYlBu_r', alpha=0.8)
q = ax1.quiver(Xv[::2,::2], Yv[::2,::2], U[::2,::2], V[::2,::2],
               velocidade[::2,::2], cmap='RdYlBu_r',
               scale=20, width=0.003)
cilindro = plt.Circle((0,0), R, color='gray', zorder=5)
ax1.add_patch(cilindro)
plt.colorbar(im, ax=ax1, label='|v| (m/s)')
ax1.set_title('Campo Vetorial — Quiver Plot\n(fluxo ao redor de cilindro)', fontsize=11)
ax1.set_aspect('equal')
ax1.set_xlim(-3, 3); ax1.set_ylim(-2, 2)

# Streamlines
ax2 = axes[1]
ax2.contourf(Xv, Yv, velocidade, levels=30, cmap='RdYlBu_r', alpha=0.8)
ax2.streamplot(Xv, Yv, U, V, color=velocidade, cmap='RdYlBu_r',
               linewidth=1.5, density=1.5, arrowsize=1.5)
cilindro2 = plt.Circle((0,0), R, color='gray', zorder=5)
ax2.add_patch(cilindro2)
ax2.set_title('Linhas de Fluxo — Streamlines\n(visão macro do escoamento)', fontsize=11)
ax2.set_aspect('equal')
ax2.set_xlim(-3, 3); ax2.set_ylim(-2, 2)

plt.suptitle('Visualização de Campo Vetorial — Escoamento ao Redor de Cilindro', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Scatter 3D com Encoding Múltiplo

Scatter plots 3D permitem representar até **5 dimensões** de dados simultaneamente usando posição (x,y,z), cor e tamanho. Isso é central em análise de dados multivariados.

In [ ]:
np.random.seed(42)
n = 300

# Simulação de dados de estrelas: posição (x,y,z), temperatura, luminosidade
clusters = [
    {'centro': (0, 0, 0),   'cor_base': 0.1, 'n': 100},  # estrelas quentes (azuis)
    {'centro': (5, 3, 2),   'cor_base': 0.6, 'n': 100},  # estrelas médias (amarelas)
    {'centro': (-3, 6, -2), 'cor_base': 0.9, 'n': 100},  # estrelas frias (vermelhas)
]

xs, ys, zs, temps, lums = [], [], [], [], []

for cl in clusters:
    cx, cy, cz = cl['centro']
    nc = cl['n']
    xs.extend(np.random.normal(cx, 1.5, nc))
    ys.extend(np.random.normal(cy, 1.5, nc))
    zs.extend(np.random.normal(cz, 1.5, nc))
    temp = np.random.normal(cl['cor_base'], 0.05, nc)
    temps.extend(temp)
    lums.extend(np.abs(np.random.normal(0.5 - 0.4*temp, 0.1, nc)))

xs, ys, zs = np.array(xs), np.array(ys), np.array(zs)
temps, lums = np.array(temps), np.array(lums)

fig = plt.figure(figsize=(15, 5))

# Scatter 3D
ax1 = fig.add_subplot(121, projection='3d')
sc = ax1.scatter(xs, ys, zs,
                  c=temps, cmap='RdYlBu_r',
                  s=lums*200 + 5,
                  alpha=0.7, edgecolors='none')
plt.colorbar(sc, ax=ax1, label='Temperatura (frio→quente)', shrink=0.6)
ax1.set_xlabel('X (pc)'); ax1.set_ylabel('Y (pc)'); ax1.set_zlabel('Z (pc)')
ax1.set_title('Catálogo de Estrelas — 5D\n(x,y,z = posição; cor = temperatura; tamanho = luminosidade)', fontsize=10)

# Diagrama HR (Hertzsprung-Russell) — Temperatura × Luminosidade
ax2 = fig.add_subplot(122)
sc2 = ax2.scatter(temps, lums, c=temps, cmap='RdYlBu_r',
                   s=60, alpha=0.7, edgecolors='gray', linewidths=0.3)
plt.colorbar(sc2, ax=ax2, label='Temperatura (index)')
ax2.set_xlabel('Índice de Temperatura (0=quente, 1=fria)', fontsize=10)
ax2.set_ylabel('Luminosidade (u.a.)', fontsize=10)
ax2.set_title('Diagrama HR simulado\n(Temperatura × Luminosidade)', fontsize=11)
ax2.invert_xaxis()  # convenção astronômica: estrelas quentes à esquerda

plt.suptitle('Visualização Multivariada — Catálogo Estelar Simulado', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Visualização de Volume (Volume Rendering Simplificado)

O **volume rendering** visualiza campos 3D (como dados de tomografia ou simulações de fluidos). Usamos fatias 2D (slices) como alternativa didática ao ray casting volumétrico.

In [ ]:
# Volume 3D simulado: duas esferas gaussianas com intensidades diferentes
size = 60
cx, cy, cz = np.mgrid[0:size, 0:size, 0:size]

# Esfera 1: alta intensidade ("nódulo" no centro-esquerdo)
r1 = np.sqrt((cx-20)**2 + (cy-30)**2 + (cz-30)**2)
esfera1 = np.exp(-r1**2 / 80) * 1.0

# Esfera 2: intensidade menor ("estrutura" à direita)
r2 = np.sqrt((cx-45)**2 + (cy-25)**2 + (cz-35)**2)
esfera2 = np.exp(-r2**2 / 50) * 0.6

# Ruído de fundo (simula tecido)
np.random.seed(7)
ruido = np.random.exponential(0.04, (size, size, size))

volume = esfera1 + esfera2 + ruido
volume = (volume - volume.min()) / (volume.max() - volume.min())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Fatias axiais (plano XY em diferentes Z)
slices_z = [10, 20, 30, 40]
for i, sz in enumerate(slices_z):
    im = axes[0, i].imshow(volume[:, :, sz], cmap='hot', vmin=0, vmax=1)
    axes[0, i].set_title(f'Fatia Axial Z={sz}', fontsize=10)
    axes[0, i].axis('off')
    plt.colorbar(im, ax=axes[0, i], fraction=0.046)

# Fatias nos 3 planos ortogonais (no centro do volume)
mid = size // 2
planos = [
    (volume[mid, :, :], 'Sagital (YZ)'),
    (volume[:, mid, :], 'Coronal (XZ)'),
    (volume[:, :, mid], 'Axial (XY)'),
]
for i, (fatia, nome) in enumerate(planos):
    im = axes[1, i].imshow(fatia, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Vista {nome}', fontsize=10)
    axes[1, i].axis('off')
    plt.colorbar(im, ax=axes[1, i], fraction=0.046)

# MIP — Maximum Intensity Projection
mip = volume.max(axis=0)  # projeção de máxima intensidade ao longo de X
im_mip = axes[1, 3].imshow(mip, cmap='hot', vmin=0, vmax=1)
axes[1, 3].set_title('MIP — Projeção de\nMáxima Intensidade', fontsize=10)
axes[1, 3].axis('off')
plt.colorbar(im_mip, ax=axes[1, 3], fraction=0.046)

plt.suptitle('Visualização de Volume 3D — Fatias e MIP (simulação de tomografia)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Comparativo Final das 4 Áreas

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

dados = [
    ['Área',              'Entrada',                  'Saída',                     'Operação Central',          'Exemplos'],
    ['Síntese de\nImagens',  'Modelo 3D,\nluz, câmera',   'Imagem renderizada',        'Rasterização / Ray Tracing', 'Jogos, CGI,\nCAD'],
    ['Processamento\nde Imagens', 'Imagem real',        'Imagem melhorada\nou métricas', 'Filtragem,\nmorfologia',   'Instagram,\nDICOM'],
    ['Visão\nComputacional', 'Imagem/vídeo real',       'Interpretação\nsemântica',   'Features + IA',              'Face ID,\nautônomos'],
    ['Visualização\nComputacional', 'Dados abstratos', 'Representação\ngráfica',     'Mapeamento visual',          'Meteorologia,\nFEM, CFD'],
]

cores_linhas = ['#2c3e50', '#c0392b', '#27ae60', '#2980b9', '#8e44ad']
cores_fonte  = ['white',   'white',   'white',   'white',   'white']

tabela = ax.table(
    cellText=dados[1:],
    colLabels=dados[0],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)

tabela.auto_set_font_size(False)
tabela.set_fontsize(10)

for (row, col), cell in tabela.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cor = cores_linhas[row]
        cell.set_facecolor(cor)
        cell.set_text_props(color='white')
    cell.set_edgecolor('white')
    cell.set_linewidth(2)

ax.set_title('Comparativo das 4 Áreas da Computação Visual', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('comparativo_areas.png', dpi=150, bbox_inches='tight')
plt.show()
print("Tabela salva em comparativo_areas.png")

## Conclusão

**Visualização Computacional** é a área que usa o sistema visual humano como canal de comunicação de dados complexos:

| Aspecto | Visualização Computacional |
|---------|---------------------------|
| **Entrada** | Dados numéricos abstratos (escalares, vetores, volumes) |
| **Saída** | Representação gráfica intuitiva |
| **Operação** | Mapeamento de dados em elementos visuais (cor, forma, posição) |
| **Exemplos** | Mapas de temperatura, simulação CFD, tomografia |

Ao contrário das outras três áreas, a Visualização Computacional não processa nem gera imagens — ela usa imagens como **linguagem** para comunicar conhecimento contido nos dados.